This data report uses the following files:
- fluxnet_shuttle_snapshot_20261116T133301.csv
- AMF_US-UMd_FLUXNET_2007-2025_v1.3_r1.zip
- EUF_AT-Neu_FLUXNET_2002-2020_v1.3_r1.zip
- EUF_RU-Fy2_FLUXNET_2015-2025_v1.3_r1.zip
- ICOS_DE-Kli_FLUXNET_2004-2025_v1.3_r1.zip
- JPF_JP-MMF_FLUXNET_2004-2011_v1.3_r1.zip

Description of the variables

| Variable | Description | Unit |
|---|---|---|
| `TIMESTAMP_START` | Start of the measurement interval | YYYYMMDDHHMM |
| `TIMESTAMP_END` | End of the measurement interval | YYYYMMDDHHMM |
| `TA_F` | Air temperature | °C |
| `SW_IN_F` | Incoming shortwave radiation | W m⁻² |
| `VPD_F` | Vapor pressure deficit | hPa |
| `P_F` | Precipitation | mm |
| `NETRAD` | Net radiation | W m⁻² |
| `G_F_MDS` | Ground heat flux | W m⁻² |
| `H_F_MDS` | Sensible heat flux | W m⁻² |
| `H_F_MDS_QC` | Quality flag for sensible heat flux | 0–1 |
| `LE_F_MDS` | Latent heat flux | W m⁻² |
| `LE_F_MDS_QC` | Quality flag for latent heat flux | 0–1 |
| `NEE_VUT_REF` | Net ecosystem exchange of CO₂ | µmol CO₂ m⁻² s⁻¹ |
| `NEE_VUT_REF_QC` | Quality flag for NEE | 0–1 |
| `GPP_NT_VUT_REF` | Gross primary production (nighttime partitioning) | µmol CO₂ m⁻² s⁻¹ |
| `RECO_NT_VUT_REF` | Ecosystem respiration (nighttime partitioning) | µmol CO₂ m⁻² s⁻¹ |

**Notes:**
- `_F`: Gap-filled data.
- `_MDS`: Marginal Distribution Sampling method.
- `_QC`: Quality-control flag.
- `VUT`: Variable u* threshold.
- `NT`: Nighttime partitioning.
- `REF`: Reference dataset.

In [120]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook"
import zipfile
import seaborn as sns
from IPython.display import display, Markdown
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from scipy.stats import linregress

## C1. Site overview

In [121]:
# Load the snapshot data
# Data directory
data_dir = Path("../data/snapshots")

# File name
file = "fluxnet_shuttle_snapshot_20261116T133301.csv"

# Combine paths
path = data_dir / file

# Read data
snap = pd.read_csv(path)


In [122]:
# get my towers and make overview table
# my towers
my_tower = ['US-UMd' ,'AT-Neu', 'RU-Fy2' ,'DE-Kli' ,'JP-MMF']
my_data = snap[snap["site_id"].isin(my_tower)]

# make overview table and print it
my_data_df = pd.DataFrame({
    "Site_ID": my_data.site_id,
    "Name": my_data.site_name,
    "Country": ["US","Japan","Germany","Russia","Austria"],
    "IGBP_class": my_data.igbp,
    "Latitude":my_data.location_lat,
    "Longitude":my_data.location_long,
    "First_year":my_data.first_year,
    "Last_year":my_data.last_year,
    "Hub": my_data.data_hub,
    "Citation":my_data.product_citation})

my_data_df

,Site_ID,Name,Country,IGBP_class,Latitude,Longitude,First_year,Last_year,Hub,Citation
311,US-UMd,UMBS Disturbance,US,DBF,45.562500,-84.697500,2007,2025,AmeriFlux,"Gil Bohrer, Christopher Gough, Aimee Classen, John Lenters, Peter Curtis (2026), AmeriFlux FLUXNET-1F US-UMd UMBS Disturbance, Ver. v1.3_r1 (2007-2025), AmeriFlux AMP, (Dataset). https://doi.org/10.17190/AMF/1881597"
444,JP-MMF,Moshiri Mixd Forest Site,Japan,MF,44.321884,142.261520,2004,2011,ICOS,"Hara, T., Kodama, Y., Nakai, T., Ohta, T., Sumida, A., Toda, M. (2026). Fluxnet Archive Product from Moshiri Mixd Forest Site, 2004–2011, FLUXNET, https://hdl.handle.net/11676/Gq5ddm9JljmhtiELU0sAXSqu"
521,DE-Kli,Klingenberg,Germany,CRO,50.893044,13.522300,2004,2025,ICOS,"Bernhofer, C., Eichelmann, U., Gruenwald, T., Hehn, M., Mauder, M., Moderow, U., Prasse, H. (2026). Fluxnet Archive Product from Klingenberg, 2004–2025, FLUXNET, https://hdl.handle.net/11676/mlrCmtpUu7ZzEbkvYEK1DpMg"
586,RU-Fy2,Fyodorovskoye dry spruce stand,Russia,ENF,56.447600,32.901880,2015,2025,ICOS,"Varlagin, A. (2026). Fluxnet Archive Product from Fyodorovskoye dry spruce stand, 2015–2025, FLUXNET, https://hdl.handle.net/11676/-FEeYNcv7UkMTJ4l72w8CLL9"
698,AT-Neu,Neustift,Austria,GRA,47.116318,11.320229,2002,2020,ICOS,"Wohlfahrt, G. (2026). Fluxnet Archive Product from Neustift, 2002–2020, FLUXNET, https://hdl.handle.net/11676/igjTOOy6rdzyQ7RPN590N98N"


In [123]:
# make a simple map
fig = px.scatter_geo(
    my_data_df,
    lat="Latitude",
    lon="Longitude",
    hover_name="Name",
    color="Site_ID",
    projection="mercator",
    scope="world"
)
fig.show()

## C2. Data coverage

In [124]:
# first load the data
keep = ["TIMESTAMP_START", "TIMESTAMP_END", "TA_F", "SW_IN_F", "VPD_F", "P_F",
        "NETRAD", "G_F_MDS", "H_F_MDS", "H_F_MDS_QC", "LE_F_MDS", "LE_F_MDS_QC",
        "NEE_VUT_REF", "NEE_VUT_REF_QC", "GPP_NT_VUT_REF", "RECO_NT_VUT_REF"]

def read_hh(zip_file):
    site = zip_file.stem.split("_")[1]
    with zipfile.ZipFile(zip_file) as z:
        hh_name = [n for n in z.namelist() if "FLUXMET_HH" in n][0]
        with z.open(hh_name) as f:
            hh = pd.read_csv(f, na_values=[-9999],
                             usecols=lambda c: c in keep,
                             dtype={"TIMESTAMP_START": str, "TIMESTAMP_END": str})
    hh["time"] = pd.to_datetime(hh["TIMESTAMP_START"], format="%Y%m%d%H%M")
    hh["site"] = site
    return hh

zip_files = sorted(Path("../data/raw").glob("*.zip"))
towers = pd.concat([read_hh(z) for z in zip_files], ignore_index=True)

In [125]:
# calculate the share of half hours in which NEE was measured per site and year
# Extract year from timestamp
towers["year"] = towers["time"].dt.year

# Calculate share of valid NEE measurements per site and year
nee_share = (
    towers
    .assign(NEE_measured=towers["NEE_VUT_REF_QC"] == 0)
    .groupby(["site", "year"])["NEE_measured"]
    .mean()
    .reset_index()
)

# Convert to percentage
nee_share["NEE_share_percent"] = nee_share["NEE_measured"] * 100


In [126]:
# make the heatmap
# Create matrix: sites = rows, years = columns
heatmap_data = nee_share.pivot(
    index="site",
    columns="year",
    values="NEE_share_percent"
)

fig = px.imshow(
    heatmap_data,
    labels=dict(
        x="Year",
        y="Site",
        color="NEE measured (%)"
    ),
    color_continuous_scale="viridis",
    aspect="auto",
    zmin=0,
    zmax=100,
    text_auto=".1f",
    title="Share of NEE Measurements"
)

fig.show()

Overall the tower in the US (US-UMd) has the best data coverage of NEE. The tower in Germany (DE-Kli) covers more years than the US tower, but has less measurments of NEE in most years.

## C3. Summary statistics

In [127]:
variables = {
    "TA_F": "°C",
    "SW_IN_F": "W m⁻²",
    "VPD_F": "hPa",
    "P_F": "mm/year",
    "NEE_VUT_REF": "µmol CO₂ m⁻² s⁻¹",
    "GPP_NT_VUT_REF": "µmol CO₂ m⁻² s⁻¹",
    "RECO_NT_VUT_REF": "µmol CO₂ m⁻² s⁻¹",
    "H_F_MDS": "W m⁻²",
    "LE_F_MDS": "W m⁻²"
}

data = towers.copy()
data["year"] = data["time"].dt.year

for site, group in data.groupby("site"):

    stats = {}

    for var, unit in variables.items():

        if var == "P_F":
            # Annual precipitation sums
            annual_sum = group.groupby("year")[var].sum(min_count=1)

            stats[f"{var} ({unit})"] = [
                annual_sum.mean(),
                annual_sum.min(),
                annual_sum.max(),
                group[var].isna().mean() * 100
            ]

        else:
            stats[f"{var} ({unit})"] = [
                group[var].mean(),
                group[var].min(),
                group[var].max(),
                group[var].isna().mean() * 100
            ]

    # Create table
    site_table = pd.DataFrame(
        stats,
        index=["Mean", "Min", "Max", "Missing (%)"]
    )

    display(Markdown(f"### Site: {site}"))
    display(site_table.round(2))

### Site: AT-Neu

,TA_F (°C),SW_IN_F (W m⁻²),VPD_F (hPa),P_F (mm/year),NEE_VUT_REF (µmol CO₂ m⁻² s⁻¹),GPP_NT_VUT_REF (µmol CO₂ m⁻² s⁻¹),RECO_NT_VUT_REF (µmol CO₂ m⁻² s⁻¹),H_F_MDS (W m⁻²),LE_F_MDS (W m⁻²)
Mean,7.01,135.59,3.13,767.58,1.32,6.19,7.48,0.67,33.28
Min,-22.35,0.00,0.00,498.60,-40.47,-37.59,0.00,-199.94,-182.64
Max,40.64,1169.87,69.45,1343.42,41.13,78.22,64.20,375.39,682.00
Missing (%),0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


### Site: DE-Kli

,TA_F (°C),SW_IN_F (W m⁻²),VPD_F (hPa),P_F (mm/year),NEE_VUT_REF (µmol CO₂ m⁻² s⁻¹),GPP_NT_VUT_REF (µmol CO₂ m⁻² s⁻¹),RECO_NT_VUT_REF (µmol CO₂ m⁻² s⁻¹),H_F_MDS (W m⁻²),LE_F_MDS (W m⁻²)
Mean,8.31,129.57,3.5,772.28,-0.25,3.47,3.23,7.08,33.96
Min,-22.76,0.00,0.0,535.10,-50.26,-40.29,0.00,-292.47,-277.80
Max,35.14,1095.00,46.1,989.33,46.11,62.07,19.48,435.32,876.06
Missing (%),0.00,0.00,0.0,0.00,2.69,2.69,2.69,1.10,2.56


### Site: JP-MMF

,TA_F (°C),SW_IN_F (W m⁻²),VPD_F (hPa),P_F (mm/year),NEE_VUT_REF (µmol CO₂ m⁻² s⁻¹),GPP_NT_VUT_REF (µmol CO₂ m⁻² s⁻¹),RECO_NT_VUT_REF (µmol CO₂ m⁻² s⁻¹),H_F_MDS (W m⁻²),LE_F_MDS (W m⁻²)
Mean,5.50,133.31,2.29,1122.76,-1.84,4.05,2.13,21.01,43.49
Min,-255.66,0.00,0.00,820.50,-75.53,-24.06,0.00,-299.90,-198.06
Max,30.46,1102.30,29.44,1596.84,26.47,77.34,18.91,752.96,727.43
Missing (%),0.00,0.00,0.00,0.00,6.53,12.57,12.57,6.47,6.48


### Site: RU-Fy2

,TA_F (°C),SW_IN_F (W m⁻²),VPD_F (hPa),P_F (mm/year),NEE_VUT_REF (µmol CO₂ m⁻² s⁻¹),GPP_NT_VUT_REF (µmol CO₂ m⁻² s⁻¹),RECO_NT_VUT_REF (µmol CO₂ m⁻² s⁻¹),H_F_MDS (W m⁻²),LE_F_MDS (W m⁻²)
Mean,6.38,112.53,3.06,592.84,0.15,4.43,4.58,16.52,29.51
Min,-30.12,0.00,0.00,324.77,-35.77,-23.45,0.00,-140.89,-62.63
Max,33.40,996.47,34.38,800.14,34.12,51.14,22.46,512.23,497.73
Missing (%),0.00,0.00,0.00,0.00,3.39,3.39,3.39,3.39,3.39


### Site: US-UMd

,TA_F (°C),SW_IN_F (W m⁻²),VPD_F (hPa),P_F (mm/year),NEE_VUT_REF (µmol CO₂ m⁻² s⁻¹),GPP_NT_VUT_REF (µmol CO₂ m⁻² s⁻¹),RECO_NT_VUT_REF (µmol CO₂ m⁻² s⁻¹),H_F_MDS (W m⁻²),LE_F_MDS (W m⁻²)
Mean,7.41,153.55,3.88,3305.21,-1.12,3.62,2.51,31.97,30.59
Min,-25.00,0.00,0.00,2715.63,-43.80,-28.81,0.00,-266.03,-392.09
Max,34.58,1128.27,39.01,4419.72,33.50,49.40,11.55,698.20,970.41
Missing (%),0.00,0.00,0.00,0.00,3.20,3.20,3.20,8.65,3.20


## C4. Time series

In [128]:
# Variables to aggregate
flux_variables = [
    "NEE_VUT_REF",
    "GPP_NT_VUT_REF",
    "RECO_NT_VUT_REF"
]

# Conversion factor
conversion_factor = 1.0378

# Create a copy of the data
daily_data = towers.copy()

# Set timestamp as index
daily_data["time"] = pd.to_datetime(daily_data["time"])

# Aggregate half-hourly data to daily means per site
daily_data = (
    daily_data
    .set_index("time")
    .groupby("site")[flux_variables]
    .resample("D")
    .mean()
    .reset_index()
)

# Convert daily mean fluxes to g C m-2 d-1
for var in flux_variables:
    daily_data[var] = daily_data[var] * conversion_factor

print(daily_data.head())

     site       time  NEE_VUT_REF  GPP_NT_VUT_REF  RECO_NT_VUT_REF
0  AT-Neu 2002-01-01    -1.379064        1.379064     1.318395e-07
1  AT-Neu 2002-01-02    -1.347209        1.347209     1.348372e-07
2  AT-Neu 2002-01-03    -1.613204        1.613204     1.209927e-07
3  AT-Neu 2002-01-04    -1.481305        1.481305     1.279892e-07
4  AT-Neu 2002-01-05    -1.063863        1.063863     1.396864e-07


In [129]:
daily_long = daily_data.melt(
    id_vars=["time", "site"],
    value_vars=flux_variables,
    var_name="Variable",
    value_name="Carbon flux (g C m⁻² d⁻¹)"
)

# Shorten variable names for the legend
daily_long["Variable"] = daily_long["Variable"].replace({
    "NEE_VUT_REF": "NEE",
    "GPP_NT_VUT_REF": "GPP",
    "RECO_NT_VUT_REF": "RECO"
})

In [130]:
fig = px.line(
    daily_long,
    x="time",
    y="Carbon flux (g C m⁻² d⁻¹)",
    color="Variable",
    facet_row="site",
    title="Daily NEE, GPP and RECO across all Flux Tower Sites",
    color_discrete_map={
        "NEE": "black",
        "GPP": "forestgreen",
        "RECO": "blue"
    }
)

fig.update_traces(
    line=dict(width=1.2),
    opacity=0.6
)

fig.update_layout(
    height=1100,
    width=1100,
    hovermode="x unified",
    legend_title="Variable"
)

fig.update_yaxes(matches=None, showticklabels=True)
fig.update_xaxes(title="Date")

fig.show()

The figure shows NEE, GPP and Reco for the sites. 

All sites show an annual cycle of NEE, where in summer typically more CO2 is captured by plants then there is respiration and therefore NEE is negative. In winter NEE revolves around zero. 

For GPP and Reco, the annual cycle is also visible. In summer GPP is typically higher than respiration, in winter respiration takes over.

There is one site that looks a bit different. AT-Neu has higher values of RECO than GPP at many time points over the year, resulting in the site being a carbon source or close to zero.

In [131]:
# Check the aggregation against the daily FLUXMET_DD file for US-UMd site
# Find ZIP file
z = list(Path("../data/raw").glob("AMF_US-UMd*.zip"))[0]

# Find FLUXMET_DD file inside ZIP
with zipfile.ZipFile(z) as archive:

    dd_name = [
        name for name in archive.namelist()
        if "FLUXMET_DD" in name
    ][0]

    # Read daily data
    with archive.open(dd_name) as f:
        dd = pd.read_csv(
            f,
            na_values="-9999",
            usecols=["TIMESTAMP", "NEE_VUT_REF","GPP_NT_VUT_REF","RECO_NT_VUT_REF"],
            dtype={"TIMESTAMP": str}
        )

# Convert timestamp to date
dd["date"] = pd.to_datetime(
    dd["TIMESTAMP"],
    format="%Y%m%d"
).dt.date


In [132]:
# Prepare daily data for comparison
daily_data["date"] = daily_data["time"].dt.date

# Select US-UMd and join with official daily data
cmp = (
    daily_data[daily_data["site"] == "US-UMd"]
    .merge(dd, on="date", suffixes=("_calculated", "_official"))
)

variables = [
    ("NEE_VUT_REF", "NEE"),
    ("GPP_NT_VUT_REF", "GPP"),
    ("RECO_NT_VUT_REF", "RECO")
]

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=["NEE", "GPP", "RECO"]
)

for i, (var, name) in enumerate(variables, start=1):

    x = cmp[f"{var}_official"]
    y = cmp[f"{var}_calculated"]

    # Scatter points
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="markers",
            marker=dict(size=4, opacity=0.4, color="steelblue"),
            showlegend=False
        ),
        row=1,
        col=i
    )

    # 1:1 reference line
    min_val = min(x.min(), y.min())
    max_val = max(x.max(), y.max())

    fig.add_trace(
        go.Scatter(
            x=[min_val, max_val],
            y=[min_val, max_val],
            mode="lines",
            line=dict(color="red", width=2),
            showlegend=False
        ),
        row=1,
        col=i
    )

    # Equal axis scaling
    fig.update_xaxes(
        title_text=f"Official {name} (g C m⁻² d⁻¹)",
        row=1,
        col=i
    )

    fig.update_yaxes(
        title_text=f"Own {name} (g C m⁻² d⁻¹)",
        scaleanchor=f"x{i}",
        scaleratio=1,
        row=1,
        col=i
    )

fig.update_layout(
    title="Comparison of calculated and official daily carbon fluxes – US-UMd",
    template="simple_white",
    width=1300,
    height=500
)

fig.show()

## C5. Diurnal cycles

The mean diurnal courses of NEE and LE were calculated separately for each site and season by averaging half-hourly measurements for each time of day. Seasonal classification was based on calendar months in the Northern Hemisphere: spring (March–May), summer (June–August), autumn (September–November), and winter (December–February). Since all five study sites are located in the Northern Hemisphere, no seasonal adjustment for the Southern Hemisphere was necessary. Hourly patterns were calculated using local standard time.

In [133]:
# Create a copy
diurnal_data = towers.copy()

# Extract month and hour
diurnal_data["month"] = diurnal_data["time"].dt.month

diurnal_data["hour"] = (
    diurnal_data["time"].dt.hour +
    diurnal_data["time"].dt.minute / 60
)

# Define seasons for the Northern Hemisphere
season_dict = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
}

diurnal_data["season"] = diurnal_data["month"].map(season_dict)

# Define season order
season_order = ["Spring", "Summer", "Autumn", "Winter"]

diurnal_data["season"] = pd.Categorical(
    diurnal_data["season"],
    categories=season_order,
    ordered=True
)

In [134]:
diurnal = (
    diurnal_data
    .groupby(["site", "season", "hour"], observed=True)
    .agg(
        NEE=("NEE_VUT_REF", "mean"),
        LE=("LE_F_MDS", "mean")
    )
    .reset_index()
)

print(diurnal.head())

     site  season  hour       NEE        LE
0  AT-Neu  Spring   0.0  6.603793  1.814574
1  AT-Neu  Spring   0.5  6.594973  1.526418
2  AT-Neu  Spring   1.0  6.648404  1.730543
3  AT-Neu  Spring   1.5  6.618302  1.393903
4  AT-Neu  Spring   2.0  6.603270  1.254760


In [135]:
diurnal_long = diurnal.melt(
    id_vars=["site", "season", "hour"],
    value_vars=["NEE", "LE"],
    var_name="Variable",
    value_name="Value"
)

In [136]:
fig = px.line(
    diurnal_long,
    x="hour",
    y="Value",
    color="site",
    facet_row="Variable",
    facet_col="season",
    category_orders={
        "season": ["Spring", "Summer", "Autumn", "Winter"],
        "Variable": ["NEE", "LE"]
    },
    labels={
        "hour": "Hour of day (local standard time)",
        "Value": "NEE (µmol CO₂ m⁻² s⁻¹) / LE (W m⁻²)",
        "site": "Site"
    },
    title="Mean Diurnal Course by Season"
)

fig.update_layout(
    template="simple_white",
    height=750,
    width=1300,
    legend=dict(
        orientation="h",
        y=1.12,
        x=0.5,
        xanchor="center"
    )
)

fig.update_xaxes(
    tickmode="array",
    tickvals=[0, 6, 12, 18, 24],
    title="Hour of day (local standard time)"
)

# Set individual y-axis labels
fig.update_yaxes(
    title_text="NEE (µmol CO₂ m⁻² s⁻¹)",
    row=2
)

fig.update_yaxes(
    title_text="LE (W m⁻²)",
    row=1
)

# Add grid to all panels
fig.update_xaxes(
    showgrid=True,
    gridcolor="lightgray",
    gridwidth=0.5
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="lightgray",
    gridwidth=0.5
)

fig.update_yaxes(matches=None)

fig.show()

## C6. My reading

In [137]:
# Select tower
site = "JP-MMF"

# Filter half-hourly data
eb = towers[
    (towers["site"] == site) &
    (towers["H_F_MDS_QC"] == 0) &
    (towers["LE_F_MDS_QC"] == 0)
].copy()

# Calculate available and turbulent energy
eb["available"] = eb["NETRAD"] - eb["G_F_MDS"]
eb["turbulent"] = eb["H_F_MDS"] + eb["LE_F_MDS"]

# Remove missing values
eb = eb.dropna(subset=["available", "turbulent"])

print(f"Site: {site}")
print(f"Number of half-hours: {len(eb)}")

Site: JP-MMF
Number of half-hours: 36207


In [138]:
# Calculate linear regression
fit = linregress(
    eb["available"],
    eb["turbulent"]
)

slope = fit.slope
intercept = fit.intercept

print(f"Slope: {slope:.2f}")
print(f"Intercept: {intercept:.2f}")
print(f"R²: {fit.rvalue**2:.3f}")
print(f"n = {len(eb)}")

Slope: 0.74
Intercept: 17.95
R²: 0.794
n = 36207


In [139]:
# Determine plot limits
min_val = min(
    eb["available"].min(),
    eb["turbulent"].min()
)

max_val = max(
    eb["available"].max(),
    eb["turbulent"].max()
)

# Create scatter plot
fig = px.scatter(
    eb,
    x="available",
    y="turbulent",
    opacity=0.1,
    labels={
        "available": "NETRAD − G (W m⁻²)",
        "turbulent": "H + LE (W m⁻²)"
    },
    title=f"Energy Balance – {site}"
)

# 1:1 reference line
fig.add_trace(
    go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode="lines",
        line=dict(color="red", width=2),
        name="1:1 line"
    )
)

# Linear regression line
x_fit = np.linspace(min_val, max_val, 100)
y_fit = intercept + slope * x_fit

fig.add_trace(
    go.Scatter(
        x=x_fit,
        y=y_fit,
        mode="lines",
        line=dict(color="blue", width=2),
        name="Linear fit"
    )
)

# Add slope and sample size
fig.add_annotation(
    x=0.03,
    y=0.97,
    xref="paper",
    yref="paper",
    text=f"Slope = {slope:.2f}<br>n = {len(eb):,}",
    showarrow=False,
    align="left",
    bgcolor="white"
)

# Equal axis scaling
fig.update_yaxes(
    scaleanchor="x",
    scaleratio=1
)

fig.update_layout(
    template="simple_white",
    width=750,
    height=700
)

fig.show()

Available energy is shown on the x-axis and is calculated as net radiation minus ground heat flux (NETRAD − G). Turbulent energy is shown on the y-axis and represents the sum of sensible and latent heat fluxes (H + LE).

Under ideal conditions, the relationship between available and turbulent energy would follow the 1:1 line, resulting in a regression slope of 1. However, due to measurement uncertainties and other factors, the energy balance is often not fully closed in eddy covariance measurements. In this case, the regression slope of 0.7 indicates that turbulent energy increases less than available energy, suggesting an underestimation of the turbulent energy fluxes relative to the available energy.

### Tower comparison

In [140]:
# Copy data and extract year
annual_data = towers.copy()
annual_data["year"] = annual_data["time"].dt.year

# Expected number of half-hours per year
annual_data["expected_hh"] = np.where(
    annual_data["time"].dt.is_leap_year, 366 * 48, 365 * 48
)

# Calculate annual NEE sums per site and year
annual_sums = (
    annual_data
    .groupby(["site", "year"])
    .agg(
        nee_sum=(
            "NEE_VUT_REF",
            lambda x: x.sum(min_count=len(x))
        ),
        n_valid=("NEE_VUT_REF", "count"),
        n_expected=("expected_hh", "first")
    )
    .reset_index()
)

# Convert to g C m-2 per year
annual_sums["nee_sum"] *= 1800 * 12.011e-6

# Keep only complete years
annual_sums = annual_sums[
    (annual_sums["n_valid"] == annual_sums["n_expected"])
]

# Calculate mean annual NEE and number of complete years
annual = (
    annual_sums
    .groupby("site")
    .agg(
        **{
            "Mean annual NEE (g C m⁻² yr⁻¹)": ("nee_sum", "mean"),
            "Complete years": ("year", "count")
        }
    )
    .reset_index()
)

In [141]:
# Extract month
daily_data["date"] = pd.to_datetime(daily_data["date"])
daily_data["month"] = daily_data["date"].dt.month

# Mean monthly GPP per site
monthly_gpp = (
    daily_data
    .groupby(["site", "month"])
    .agg(gpp=("GPP_NT_VUT_REF", "mean"))
    .reset_index()
)

# Seasonal amplitude = maximum - minimum monthly mean
seasonal = (
    monthly_gpp
    .groupby("site")
    .agg(
        **{
            "GPP seasonal amplitude (g C m⁻² d⁻¹)": (
                "gpp",
                lambda x: x.max() - x.min()
            )
        }
    )
    .reset_index()
)


In [142]:
# gapfilled NEE
gapfilled = (
    towers
    .groupby("site")
    .agg(
        **{
            "Gap-filled NEE (%)": (
                "NEE_VUT_REF_QC",
                lambda x: (~x.eq(0)).mean() * 100
            )
        }
    )
    .reset_index()
)


In [143]:
overview = (
    annual
    .merge(seasonal, on="site", how="left")
    .merge(gapfilled, on="site", how="left")
    .rename(columns={"site": "Site"})
)

overview

,Site,Mean annual NEE (g C m⁻² yr⁻¹),Complete years,GPP seasonal amplitude (g C m⁻² d⁻¹),Gap-filled NEE (%)
0,AT-Neu,501.240344,19,14.013215,75.685038
1,DE-Kli,-94.882144,18,11.341217,61.304598
2,JP-MMF,-689.022124,6,9.510167,79.555242
3,RU-Fy2,49.864802,9,12.430440,65.899805
4,US-UMd,-416.457351,16,10.608104,48.317423


The overview table shows that the tower in Japan (JP-MMF) with -689 g C m⁻² yr⁻¹ takes up most carbon of the towers. As already seen from the annual cycles, the tower in Austria (AT-Neu) releases most carbon 501 g C m⁻² yr⁻¹.

The GPP seasonal cycle is biggest in Austria, but Russia, the US and Germany follow closely. 

As also seen in C2, GAP-filled NEE is the lowest for the US tower and relatively high for the other towers.

Which suprises me most is the that the tower in Austria is a large carbon source rather than a carbon sink. Also the tower in Russia is a carbon source.

## Data and citation

Site data. Each site’s data citation, taken from the snapshot file "fluxnet_shuttle_snapshot_20261116T133301.csv"

- **US-UMd** Gil Bohrer, Christopher Gough, Aimee Classen, John Lenters, Peter Curtis (2026), AmeriFlux FLUXNET-1F US-UMd UMBS Disturbance, Ver. v1.3_r1 (2007-2025), AmeriFlux AMP, (Dataset). https://doi.org/10.17190/AMF/1881597
- **JP-MMF** Hara, T., Kodama, Y., Nakai, T., Ohta, T., Sumida, A., Toda, M. (2026). Fluxnet Archive Product from Moshiri Mixd Forest Site, 2004–2011, FLUXNET, https://hdl.handle.net/11676/Gq5ddm9JljmhtiELU0sAXSqu
- **DE-Kli** Bernhofer, C., Eichelmann, U., Gruenwald, T., Hehn, M., Mauder, M., Moderow, U., Prasse, H. (2026). Fluxnet Archive Product from Klingenberg, 2004–2025, FLUXNET, https://hdl.handle.net/11676/mlrCmtpUu7ZzEbkvYEK1DpMg
- **RU-Fy2** Varlagin, A. (2026). Fluxnet Archive Product from Fyodorovskoye dry spruce stand, 2015–2025, FLUXNET, https://hdl.handle.net/11676/-FEeYNcv7UkMTJ4l72w8CLL9
- **AT-Neu** Wohlfahrt, G. (2026). Fluxnet Archive Product from Neustift, 2002–2020, FLUXNET, https://hdl.handle.net/11676/igjTOOy6rdzyQ7RPN590N98N
- Pastorello et al. (2020), Scientific Data 7, 225, https://doi.org/10.1038/s41597-020-0534-3

Fluxnet acknowledgement:

FLUXNET data products were produced and harmonized by eddy covariance regional networks and data processing centers, including AmeriFlux, ChinaFlux, European Fluxes Database, ICOS, JapanFlux, KoFlux, OzFlux, SAEON, and TERN. These products also include a modified version of ERA5 hourly data provided by the Copernicus Climate Change Service.
